# 04 — SparkSQL: las mismas operaciones en SQL

Si venís de SQL, esto es para vos. Cualquier DataFrame se puede
registrar como vista temporal y consultarse con SQL puro.

In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .appName("04_spark_sql")
    .master("spark://spark-master:7077")
    .getOrCreate()
)

df = spark.read.parquet("/home/jovyan/work/data/sample/ventas/ventas_parquet")
df.createOrReplaceTempView("ventas")

In [ ]:
spark.sql("SELECT ciudad, COUNT(*) AS n FROM ventas GROUP BY ciudad").show()

In [ ]:
spark.sql("""
    SELECT
        ciudad,
        producto,
        COUNT(*) AS n_ventas,
        ROUND(AVG(precio), 2) AS precio_promedio,
        SUM(cantidad * precio) AS ingreso_total
    FROM ventas
    WHERE precio > 1000
    GROUP BY ciudad, producto
    HAVING COUNT(*) > 100
    ORDER BY ingreso_total DESC
    LIMIT 20
""").show()

## CTE (Common Table Expressions)

In [ ]:
spark.sql("""
    WITH resumen AS (
        SELECT
            ciudad,
            producto,
            COUNT(*) AS n_ventas,
            AVG(precio) AS precio_promedio
        FROM ventas
        GROUP BY ciudad, producto
    )
    SELECT * FROM resumen WHERE precio_promedio > 30000 ORDER BY precio_promedio DESC
""").show(10)

## Window functions en SQL

In [ ]:
spark.sql("""
    SELECT
        ciudad,
        producto,
        precio,
        ROW_NUMBER() OVER (PARTITION BY ciudad ORDER BY precio DESC) AS rank
    FROM ventas
""").filter("rank <= 3").orderBy("ciudad", "rank").show(15)

## Cuándo usar SQL vs DataFrame API

Las dos compilan al **mismo plan lógico**. La diferencia es la
ergonomía:

| Caso | Preferí |
|------|---------|
| Análisis exploratorio de un data scientist | SQL |
| Pipeline ETL con lógica condicional compleja | DataFrame API |
| Querys que tu equipo ya tiene en dbt / warehouse | SQL |
| Construcción dinámica de columnas según config | DataFrame API |

Mezclá sin culpa: muchas veces la lógica principal está en SQL y un
Python wrapper hace la lectura/escritura.